## 1. Configuration et Imports

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
import os
from datetime import datetime

In [ ]:
# Création de la session Spark avec support Delta Lake
spark = SparkSession.builder \
    .appName("IoT_JSON_to_Delta_Bronze") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.sql.streaming.checkpointLocation", "/tmp/checkpoints") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print(f"Spark version: {spark.version}")

## 2. Définition du Schéma des Données IoT

Les données proviennent de capteurs intelligents dans des bâtiments :
- `sensor_id` : Identifiant unique du capteur
- `building_id` : Identifiant du bâtiment
- `timestamp` : Horodatage de la mesure
- `temperature` : Température en °C
- `humidity` : Humidité en %
- `energy_consumption` : Consommation d'énergie en kWh
- `anomaly_detected` : Détection d'anomalie (boolean)

In [ ]:
# Schéma des données de capteurs IoT
sensor_schema = StructType([
    StructField("sensor_id", StringType(), False),
    StructField("building_id", StringType(), False),
    StructField("timestamp", TimestampType(), False),
    StructField("temperature", DoubleType(), True),
    StructField("humidity", DoubleType(), True),
    StructField("energy_consumption", DoubleType(), True),
    StructField("anomaly_detected", BooleanType(), True)
])

## 3. Configuration des Chemins

Organisation des répertoires selon l'architecture Médaillon :
- **Source** : Répertoire contenant les fichiers JSON en entrée
- **Bronze** : Données brutes stockées dans Delta Lake
- **Checkpoint** : Point de sauvegarde pour la tolérance aux pannes

In [ ]:
# Configuration des chemins
BASE_PATH = "/tmp/iot_pipeline"
SOURCE_PATH = f"{BASE_PATH}/source/sensor_data"
BRONZE_PATH = f"{BASE_PATH}/delta/bronze/sensor_data"
CHECKPOINT_PATH = f"{BASE_PATH}/checkpoints/bronze_sensor_data"

# Création des répertoires si nécessaire
os.makedirs(SOURCE_PATH, exist_ok=True)
os.makedirs(os.path.dirname(BRONZE_PATH), exist_ok=True)
os.makedirs(CHECKPOINT_PATH, exist_ok=True)

print(f"Source Path: {SOURCE_PATH}")
print(f"Bronze Path: {BRONZE_PATH}")
print(f"Checkpoint Path: {CHECKPOINT_PATH}")

## 4. Génération de Données de Test

Création d'un jeu de données JSON pour simuler les données des capteurs

In [ ]:
import json
import random
from datetime import datetime, timedelta

def generate_sample_data(num_records=100, file_index=1):
    """Génère des données de test au format JSON"""
    data = []
    base_time = datetime.now()
    
    buildings = ["BLD001", "BLD002", "BLD003", "BLD004", "BLD005"]
    
    for i in range(num_records):
        record = {
            "sensor_id": f"SENSOR_{random.randint(1, 50):03d}",
            "building_id": random.choice(buildings),
            "timestamp": (base_time + timedelta(seconds=i)).isoformat(),
            "temperature": round(random.uniform(18.0, 28.0), 2),
            "humidity": round(random.uniform(30.0, 70.0), 2),
            "energy_consumption": round(random.uniform(0.5, 5.0), 2),
            "anomaly_detected": random.random() < 0.05  # 5% d'anomalies
        }
        data.append(record)
    
    # Écriture dans un fichier JSON
    filename = f"{SOURCE_PATH}/sensor_data_{file_index:03d}.json"
    with open(filename, 'w') as f:
        for record in data:
            f.write(json.dumps(record) + '\n')
    
    print(f"Généré {num_records} enregistrements dans {filename}")
    return filename

# Génération du premier lot de données
generate_sample_data(100, 1)

## 5. Lecture du Flux JSON

Utilisation de `readStream` pour lire les fichiers JSON de manière continue

In [ ]:
# Lecture du flux de données JSON
raw_stream = spark.readStream \
    .format("json") \
    .schema(sensor_schema) \
    .option("maxFilesPerTrigger", 1) \
    .load(SOURCE_PATH)

print("Flux de lecture configuré")
print(f"Schema: {raw_stream.schema}")

## 6. Transformations Bronze

Transformations basiques appliquées aux données brutes :
- Ajout d'une colonne `ingestion_time` pour tracer l'horodatage d'ingestion
- Filtrage des valeurs nulles critiques
- Filtrage des valeurs aberrantes (température et humidité hors plage)
- Projection des colonnes pertinentes

In [ ]:
# Transformations Bronze : nettoyage et enrichissement
bronze_stream = raw_stream \
    .filter(col("sensor_id").isNotNull()) \
    .filter(col("building_id").isNotNull()) \
    .filter(col("timestamp").isNotNull()) \
    .filter((col("temperature").between(-50, 60)) | col("temperature").isNull()) \
    .filter((col("humidity").between(0, 100)) | col("humidity").isNull()) \
    .withColumn("ingestion_time", current_timestamp()) \
    .select(
        "sensor_id",
        "building_id",
        "timestamp",
        "temperature",
        "humidity",
        "energy_consumption",
        "anomaly_detected",
        "ingestion_time"
    )

print("Transformations Bronze configurées")

## 7. Écriture dans Delta Lake (Bronze)

Configuration de l'écriture en streaming vers Delta Lake :
- **Format** : Delta Lake pour ACID et versioning
- **Mode de sortie** : `append` (ajout des nouvelles données)
- **Checkpoint** : Garantie de tolérance aux pannes
- **Trigger** : `processingTime` pour traiter les données toutes les 10 secondes

In [ ]:
# Écriture en streaming vers Delta Lake Bronze
query = bronze_stream.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", CHECKPOINT_PATH) \
    .trigger(processingTime="10 seconds") \
    .start(BRONZE_PATH)

print(f"Streaming query démarré: {query.name}")
print(f"Query ID: {query.id}")
print(f"Status: {query.status}")

## 8. Surveillance du Flux

Vérification de l'état du streaming et des métriques

In [ ]:
# Afficher le statut du streaming
import time

for i in range(3):
    time.sleep(5)
    print(f"\n--- Itération {i+1} ---")
    print(f"Status: {query.status}")
    print(f"Recent Progress:")
    if query.recentProgress:
        latest = query.recentProgress[-1]
        print(f"  - Batch: {latest.get('batchId', 'N/A')}")
        print(f"  - Num Input Rows: {latest.get('numInputRows', 0)}")
        print(f"  - Input Rows Per Second: {latest.get('inputRowsPerSecond', 0)}")
        print(f"  - Process Rows Per Second: {latest.get('processedRowsPerSecond', 0)}")

## 9. Ajout de Nouvelles Données (Simulation)

Génération de nouveaux fichiers JSON pour voir le streaming en action

In [ ]:
# Générer plusieurs lots de données supplémentaires
for i in range(2, 4):
    generate_sample_data(50, i)
    print(f"Lot {i} généré")
    time.sleep(2)

In [ ]:
# Attendre le traitement et vérifier le statut
time.sleep(15)
print(f"\n--- Statut Final ---")
print(f"Is Active: {query.isActive}")
print(f"Recent Progress Count: {len(query.recentProgress)}")

## 10. Vérification des Données Bronze

Lecture batch des données Delta Lake pour validation

In [ ]:
# Lire les données Bronze en mode batch
bronze_df = spark.read.format("delta").load(BRONZE_PATH)

print(f"Nombre total d'enregistrements dans Bronze: {bronze_df.count()}")
print("\nAperçu des données Bronze:")
bronze_df.show(10, truncate=False)

In [ ]:
# Statistiques par bâtiment
print("\nStatistiques par bâtiment:")
bronze_df.groupBy("building_id").agg(
    count("*").alias("total_records"),
    avg("temperature").alias("avg_temperature"),
    avg("humidity").alias("avg_humidity"),
    sum("energy_consumption").alias("total_energy"),
    sum(when(col("anomaly_detected"), 1).otherwise(0)).alias("anomalies_count")
).show()

In [ ]:
# Détection des anomalies
print("\nEnregistrements avec anomalies détectées:")
bronze_df.filter(col("anomaly_detected") == True).show(5, truncate=False)

## 11. Historique Delta Lake

Vérification des versions et de l'historique des transactions

In [ ]:
# Historique des transactions Delta
from delta.tables import DeltaTable

delta_table = DeltaTable.forPath(spark, BRONZE_PATH)
print("\nHistorique des transactions Delta:")
delta_table.history().select("version", "timestamp", "operation", "operationMetrics").show(10, truncate=False)

## 12. Arrêt du Streaming

Arrêt propre du query de streaming

In [ ]:
# Arrêter le streaming query
query.stop()
print("Streaming query arrêté")

# Vérifier l'état
time.sleep(2)
print(f"Is Active: {query.isActive}")

## 13. Nettoyage (Optionnel)

In [ ]:
# Décommenter pour nettoyer les données de test
# import shutil
# shutil.rmtree(BASE_PATH, ignore_errors=True)
# print(f"Répertoire {BASE_PATH} supprimé")

## Concepts Clés Illustrés

### 1. **Streaming Structuré**
- Traitement de flux comme des tables infinies
- API unifiée pour batch et streaming

### 2. **Checkpointing**
- Sauvegarde de l'état du streaming
- Reprise après échec exactement au même point
- Garantie de tolérance aux pannes

### 3. **Mode de Sortie : Append**
- Seules les nouvelles lignes sont écrites
- Idéal pour les données immuables (Bronze)

### 4. **Triggers**
- `processingTime` : traitement à intervalles réguliers (10s)
- Autres options : `once`, `continuous`

### 5. **Delta Lake (Bronze)**
- Transactions ACID
- Versioning et time travel
- Données brutes avec métadonnées d'ingestion

### 6. **Architecture Médaillon - Bronze**
- Stockage des données brutes
- Transformations minimales (validation, filtrage)
- Source de vérité pour les couches suivantes (Silver, Gold)